# 🎓 Week 2 Internship Task – Student Performance Data Analysis

**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Streamlit

---

## Outline
1. Dataset Collection
2. Data Cleaning
3. Data Analysis
4. Data Visualization
5. Streamlit Dashboard (see `app.py`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✔')

## Part 1 – Dataset

We use a synthetically generated **Student Performance** dataset (1000+ records) that simulates real-world data from a Kaggle-style source. It contains subject scores, attendance, study hours, demographics, and socioeconomic indicators.

In [ ]:
df_raw = pd.read_csv('../data/student_performance.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

## Part 2 – Data Cleaning

In [ ]:
# 1. Inspect
print('Missing values:')
print(df_raw.isnull().sum())
print(f'\nDuplicates: {df_raw.duplicated().sum()}')

In [ ]:
df = df_raw.copy()

# 2. Remove duplicates
df.drop_duplicates(inplace=True)
print(f'After dedup: {df.shape}')

# 3. Fill missing values
score_cols = ['math_score','science_score','english_score','history_score','art_score']
for col in score_cols:
    df[col].fillna(df[col].median(), inplace=True)
df['study_hours_per_day'].fillna(df['study_hours_per_day'].mean(), inplace=True)
df['attendance_percent'].fillna(df['attendance_percent'].mean(), inplace=True)
df['parent_education'].fillna('Unknown', inplace=True)

print(f'Missing after fill: {df.isnull().sum().sum()}')

In [ ]:
# 4. Rename columns
df.rename(columns={
    'math_score':'Math','science_score':'Science','english_score':'English',
    'history_score':'History','art_score':'Art',
    'study_hours_per_day':'Study_Hours','attendance_percent':'Attendance',
    'parent_education':'Parent_Edu','internet_access':'Internet',
    'extracurricular':'Extracurricular'
}, inplace=True)

# 5. Fix dtypes
df['student_id'] = df['student_id'].astype(str)
df['age'] = df['age'].astype(int)

# 6. Derived column
SCORE_COLS = ['Math','Science','English','History','Art']
df['Avg_Score'] = df[SCORE_COLS].mean(axis=1).round(2)

# 7. Drop unnecessary column
df.drop(columns=['name'], inplace=True, errors='ignore')

print('Clean dataset:')
df.info()

## Part 3 – Data Analysis

In [ ]:
print(f'Total students: {len(df)}')
print(f"\nTop performer: {df.loc[df['Avg_Score'].idxmax(), 'student_id']} — {df['Avg_Score'].max():.2f}")
print(f"Low performer: {df.loc[df['Avg_Score'].idxmin(), 'student_id']} — {df['Avg_Score'].min():.2f}")
df[SCORE_COLS + ['Avg_Score','Study_Hours','Attendance']].describe().round(2)

In [ ]:
print('Subject-wise averages:')
df[SCORE_COLS].mean().sort_values(ascending=False).round(2)

In [ ]:
print('Grade-wise average:')
df.groupby('grade')['Avg_Score'].mean().sort_values(ascending=False).round(2)

In [ ]:
print('Correlation – Study Hours & Avg Score:')
df[['Study_Hours','Avg_Score']].corr()

## Part 4 – Data Visualization

In [ ]:
PALETTE = ['#264653','#2a9d8f','#e9c46a','#f4a261','#e76f51']
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})

# 1. Bar – Subject Averages
fig, ax = plt.subplots(figsize=(9,5))
avgs = df[SCORE_COLS].mean().sort_values(ascending=False)
bars = ax.bar(avgs.index, avgs.values, color=PALETTE, edgecolor='white', width=0.55)
for bar,val in zip(bars,avgs.values):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'{val:.1f}',ha='center',va='bottom',fontweight='bold')
ax.set_title('Average Score by Subject', fontsize=15, fontweight='bold')
ax.set_ylim(0,105)
plt.tight_layout(); plt.show()

In [ ]:
# 2. Line – Study Hours Trend
df['Study_Bin'] = pd.cut(df['Study_Hours'], bins=8)
trend = df.groupby('Study_Bin')['Avg_Score'].mean().dropna()
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(range(len(trend)), trend.values, color='#2a9d8f', linewidth=2.5,
        marker='o', markersize=8, markerfacecolor='white', markeredgewidth=2.5)
ax.fill_between(range(len(trend)), trend.values, alpha=0.15, color='#2a9d8f')
ax.set_xticks(range(len(trend)))
ax.set_xticklabels([str(b) for b in trend.index], rotation=30, ha='right', fontsize=8)
ax.set_title('Study Hours vs Average Score', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 3. Pie – Gender
fig, ax = plt.subplots(figsize=(7,6))
gc = df['gender'].value_counts()
wedges,texts,autotexts = ax.pie(gc,labels=gc.index,autopct='%1.1f%%',colors=['#2a9d8f','#e76f51'],
       startangle=140,pctdistance=0.80,wedgeprops=dict(edgecolor='white',linewidth=2))
for t in autotexts: t.set_fontsize(13); t.set_fontweight('bold')
ax.set_title('Gender Distribution', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 4. Histogram – Avg Score
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(df['Avg_Score'], bins=25, color='#2a9d8f', edgecolor='white', alpha=0.85)
ax.axvline(df['Avg_Score'].mean(), color='#e76f51', linewidth=2, linestyle='--', label=f"Mean: {df['Avg_Score'].mean():.1f}")
ax.axvline(df['Avg_Score'].median(), color='#264653', linewidth=2, linestyle=':', label=f"Median: {df['Avg_Score'].median():.1f}")
ax.legend(fontsize=11)
ax.set_title('Average Score Distribution', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 5. Heatmap
fig, ax = plt.subplots(figsize=(9,7))
num_cols = SCORE_COLS + ['Avg_Score','Study_Hours','Attendance']
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix,dtype=bool))
sns.heatmap(corr_matrix,mask=mask,annot=True,fmt='.2f',cmap='YlGnBu',
            linewidths=0.5,ax=ax,annot_kws={'size':9},cbar_kws={'shrink':0.8})
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 6. Box – Grade
fig, ax = plt.subplots(figsize=(9,5))
sns.boxplot(data=df,x='grade',y='Avg_Score',order=['9th','10th','11th','12th'],
            palette=PALETTE,ax=ax,linewidth=1.4)
ax.set_title('Avg Score by Grade', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## Part 5 – Streamlit Dashboard

Run the interactive dashboard:

```bash
streamlit run app.py
```

The dashboard includes:
- Dataset preview with download button
- Statistical summary tabs
- Interactive filtering (sidebar)
- 8+ visualizations
- Dynamic comparison chart (Bonus)